# VoiceGuard — train on Kaggle

HF data → content-matched MMS-TTS fakes → manifests → RawBoost → fine-tune wav2vec2 +
AASIST → evaluate. Output: `/kaggle/working/aasist_indicw2v.pt`.

**Accelerator → `GPU T4 x2`** (P100 is incompatible with Kaggle's PyTorch).
**Persistence → `Files only`.**  Run Cell 1 once interactively to confirm the GPU, then
**Save Version → Save & Run All (Commit)**.

Raw data + HF cache go to `/kaggle/temp` (60 GB, wiped per session); only the checkpoint
lands in `/kaggle/working` (kept) — so a killed run resumes training from the last epoch
after a ~15 min re-download.

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'
try:
    _ = (torch.randn(64, 64, device='cuda') @ torch.randn(64, 64, device='cuda')).sum().item()
except RuntimeError as e:
    raise SystemExit(f'GPU not usable ({e}) — set Accelerator to GPU T4 x2')
print('OK:', torch.cuda.get_device_name(0))
!pip -q install -U 'transformers>=4.44' 'datasets>=2.20' huggingface_hub soundfile librosa pyyaml scipy 2>/dev/null | tail -1

In [ ]:
REPO_URL = 'https://github.com/Deva-996/VoiceGuard'
HF_TOKEN = ''   # optional: enables ai4bharat/indicwav2vec-hindi (accept its licence first)

os.environ['HF_HOME'] = '/kaggle/temp/hf'          # keep the HF cache off the 20 GB disk
os.makedirs('/kaggle/temp/vgdata', exist_ok=True)
if HF_TOKEN: os.environ['HF_TOKEN'] = HF_TOKEN

%cd /kaggle/working
!rm -rf voiceguard && git clone --depth 1 {REPO_URL} voiceguard
%cd voiceguard
!rm -rf data && ln -s /kaggle/temp/vgdata data
!df -h /kaggle/temp /kaggle/working | tail -2; ls

In [ ]:
FRONTEND = 'facebook/wav2vec2-xls-r-300m'   # or ai4bharat/indicwav2vec-hindi (needs HF_TOKEN)
EPOCHS   = 16
UNFREEZE = 5        # frontend frozen for N epochs, then fine-tuned (the quality lever)
LOSS     = 'oc_softmax'
FAKE_N   = 900
LIMIT    = None     # e.g. 400 for a quick smoke run first

import yaml, pathlib
p = pathlib.Path('training/config_train.yaml'); cfg = yaml.safe_load(p.read_text())
cfg['device'] = 'cuda'
cfg['frontend'].update(model_id=FRONTEND, layer=-1, stage2_unfreeze_epoch=UNFREEZE)
cfg['loss']['name'] = LOSS
cfg['epochs'] = EPOCHS
cfg['num_workers'] = 2
cfg['checkpoint']['out'] = '/kaggle/working/aasist_indicw2v.pt'
p.write_text(yaml.safe_dump(cfg, sort_keys=False)); print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
args = f'--config training/config_train.yaml --fake-n {FAKE_N} --epochs {EPOCHS}'
if LIMIT: args += f' --limit {LIMIT}'
if os.path.exists('data/manifests/train.tsv'): args += ' --skip-download --skip-fakes'
!python -m training.pipeline_run {args}

In [ ]:
import torch
c = torch.load('/kaggle/working/aasist_indicw2v.pt', map_location='cpu', weights_only=False)
print('dev_eer', c.get('dev_eer'), '| epoch', c.get('epoch'), '| frontend', c.get('frontend_model_id'),
      '| finetuned', c.get('frontend') is not None)
!ls -la /kaggle/working/*.pt /kaggle/working/*.tsv 2>/dev/null